# 🚀 GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/09_transformer/gpt/gpt.ipynb)

In this notebook, we'll walk through the steps required to train your own GPT model on the wine review dataset

In [1]:
# Setup for Google Colab and local environments
import os
import sys

# Python 3.12+ compatibility shim for 'imp' (used by legacy autoreload)
if "imp" not in sys.modules:
    try:
        import imp
    except ImportError:
        import importlib
        import types

        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

# 1. If running in Google Colab, clone the repository and navigate into it
if "google.colab" in sys.modules:
    if os.path.exists("/content"):
        %cd /content
    if not os.path.exists("Generative_Deep_Learning_2nd_Edition"):
        !git clone https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition.git
    %cd /content/Generative_Deep_Learning_2nd_Edition
    if os.path.abspath(".") not in sys.path:
        sys.path.append(os.path.abspath("."))

# 2. Add repository root to sys.path if running locally outside repository root
repo_root = None
for candidate in [
    os.path.abspath(os.path.join(os.getcwd(), "../../..")),
    os.path.abspath(os.getcwd()),
    os.path.abspath("/content/Generative_Deep_Learning_2nd_Edition"),
    "/app",
]:
    if os.path.exists(os.path.join(candidate, "notebooks")):
        repo_root = candidate
        break
if repo_root is None:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

if repo_root not in sys.path:
    sys.path.append(repo_root)


/content
Cloning into 'Generative_Deep_Learning_2nd_Edition'...
remote: Enumerating objects: 653, done.
remote: Counting objects: 100% (347/347), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 653 (delta 225), reused 203 (delta 203), pack-reused 306 (from 1)
Receiving objects: 100% (653/653), 37.06 MiB | 17.26 MiB/s, done.
Resolving deltas: 100% (375/375), done.
/content/Generative_Deep_Learning_2nd_Edition


The code is adapted from the excellent [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) created by Apoorv Nandan available on the Keras website.

In [2]:
# Python 3.12+ compatibility shim for 'imp' (used by legacy autoreload)
import sys
if "imp" not in sys.modules:
    try:
        import imp
    except ImportError:
        import importlib
        import types

        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

# Enable autoreload if available and supported
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass

import os
import numpy as np
import json
import re
import string
from IPython.display import display, HTML

import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")


TensorFlow version: 2.21.0
GPU available: True


## 0. Parameters <a name="parameters"></a>

In [3]:
VOCAB_SIZE = 10000
MAX_LEN = 80
EMBEDDING_DIM = 256
KEY_DIM = 256
N_HEADS = 2
FEED_FORWARD_DIM = 256
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 5

## 1. Load the data <a name="load"></a>

In [4]:
# Resolve dataset path across Docker, local, Kaggle, or Colab environments
data_dir = None
candidate_dirs = [
    os.path.join(repo_root, "data", "wine-reviews"),
    "/app/data/wine-reviews",
    os.path.abspath("./data/wine-reviews"),
    os.path.abspath("../../../data/wine-reviews"),
    "/kaggle/input/wine-reviews",
    "/content/data/wine-reviews",
]

for candidate in candidate_dirs:
    if os.path.exists(os.path.join(candidate, "winemag-data-130k-v2.json")):
        data_dir = candidate
        break

if data_dir is None:
    data_dir = os.path.join(repo_root, "data", "wine-reviews")

json_path = os.path.join(data_dir, "winemag-data-130k-v2.json")

# If dataset is missing, automatically download from Kaggle (via kagglehub or Kaggle API)
if not os.path.exists(json_path):
    print(f"Dataset not found at {json_path}. Attempting to download 'zynicide/wine-reviews'...")
    downloaded = False
    try:
        import kagglehub
        import shutil

        dl_path = kagglehub.dataset_download("zynicide/wine-reviews")
        candidate_json = os.path.join(dl_path, "winemag-data-130k-v2.json")
        if os.path.exists(candidate_json):
            os.makedirs(data_dir, exist_ok=True)
            shutil.copy2(candidate_json, json_path)
            downloaded = True
            print(f"Downloaded and copied dataset to {json_path}")
    except Exception:
        pass

    if not downloaded and not os.path.exists(json_path):
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
            import zipfile

            if os.path.exists(os.path.join(repo_root, "kaggle.json")):
                os.environ["KAGGLE_CONFIG_DIR"] = repo_root

            api = KaggleApi()
            api.authenticate()

            os.makedirs(data_dir, exist_ok=True)
            parent_dir = os.path.dirname(data_dir)
            zip_path = os.path.join(parent_dir, "wine-reviews.zip")
            print("Downloading zynicide/wine-reviews via Kaggle API...")
            api.dataset_download_files("zynicide/wine-reviews", path=parent_dir, unzip=False)
            print("Extracting files...")
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(data_dir)
            if os.path.exists(zip_path):
                os.remove(zip_path)
            print("Download and extraction complete!")
        except Exception as e:
            raise FileNotFoundError(
                f"Could not find or automatically download 'winemag-data-130k-v2.json' at '{json_path}'.\n"
                f"Please ensure kagglehub or Kaggle API is set up, or run: bash scripts/download.sh wines\n"
                f"Error details: {e}"
            )

# Load the full dataset
with open(json_path, encoding="utf-8") as json_data:
    wine_data = json.load(json_data)


Dataset not found at /content/Generative_Deep_Learning_2nd_Edition/data/wine-reviews/winemag-data-130k-v2.json. Attempting to download 'zynicide/wine-reviews'...
Using Colab cache for faster access to the 'wine-reviews' dataset.
Downloaded and copied dataset to /content/Generative_Deep_Learning_2nd_Edition/data/wine-reviews/winemag-data-130k-v2.json


In [5]:
wine_data[10]

{'points': '87',
 'title': 'Kirkland Signature 2011 Mountain Cuvée Cabernet Sauvignon (Napa Valley)',
 'description': 'Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility.',
 'taster_name': 'Virginie Boone',
 'taster_twitter_handle': '@vboone',
 'price': 19,
 'designation': 'Mountain Cuvée',
 'variety': 'Cabernet Sauvignon',
 'region_1': 'Napa Valley',
 'region_2': 'Napa',
 'province': 'California',
 'country': 'US',
 'winery': 'Kirkland Signature'}

In [6]:
# Filter the dataset
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]

In [7]:
# Count the recipes
n_wines = len(filtered_data)
print(f"{n_wines} recipes loaded")

129907 recipes loaded


In [8]:
example = filtered_data[25]
print(example)

wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard-designated Pinot that hails from a high-elevation site. Small in production, it offers intense, full-bodied raspberry and blackberry steeped in smoky spice and smooth texture.


## 2. Tokenize the data <a name="tokenize"></a>

In [9]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [10]:
# Display an example of a recipe
example_data = text_data[25]
example_data

'wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard - designated Pinot that hails from a high - elevation site . Small in production , it offers intense , full - bodied raspberry and blackberry steeped in smoky spice and smooth texture . '

In [11]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [12]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [13]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [14]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: :
3: ,
4: .
5: and
6: the
7: wine
8: a
9: of


In [15]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[   7   10    2   20    2   29    2   43   62    2   55    5  243 4145
  453  634   26    9  497  499  667   17   12  142   14 2214   43   25
 2484   32    8  223   14 2213  948    4  594   17  987    3   15   75
  237    3   64   14   82   97    5   74 2633   17  198   49    5  125
   77    4    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0]


## 3. Create the Training Set <a name="create"></a>

In [16]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = text_ds.map(prepare_inputs)

In [17]:
example_input_output = train_ds.take(1).get_single_element()

In [18]:
# Example Input
example_input_output[0][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([   7,   10,    2,   40,    2,  144,    2,  163,    2,  688, 1653,
          3,  417,    3,   78,   14,  745,   59,    5,  789,   49,   26,
       1322,   17,    6,  283,    4,    6,   64,   14,   82,   28,  691,
        116,   38,   37,   36,    3,  276,   97,    3,  472,    5,  220,
        167,  102,    3,  388,   34,    4,  176,   18,  608, 2886,    6,
        875,    9,  261,   23,    6,   31,   33,    6,   93,   22,  755,
         15,   17, 2119,    4,   35, 2669,    4,    0,    0,    0,    0,
          0,    0,    0])>

In [19]:
# Example Output (shifted by one token)
example_input_output[1][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([  10,    2,   40,    2,  144,    2,  163,    2,  688, 1653,    3,
        417,    3,   78,   14,  745,   59,    5,  789,   49,   26, 1322,
         17,    6,  283,    4,    6,   64,   14,   82,   28,  691,  116,
         38,   37,   36,    3,  276,   97,    3,  472,    5,  220,  167,
        102,    3,  388,   34,    4,  176,   18,  608, 2886,    6,  875,
          9,  261,   23,    6,   31,   33,    6,   93,   22,  755,   15,
         17, 2119,    4,   35, 2669,    4,    0,    0,    0,    0,    0,
          0,    0,    0])>

## 5. Create the causal attention mask function <a name="causal"></a>

In [20]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)


np.transpose(causal_attention_mask(1, 10, 10, dtype=tf.int32)[0])

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [21]:
class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return (self.ln_2(out1 + ffn_output), attention_scores)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

## 7. Create the Token and Position Embedding <a name="embedder"></a>

In [22]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim, **kwargs):
        super(TokenAndPositionEmbedding, self).__init__(**kwargs)
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [23]:
inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x, attention_scores = TransformerBlock(
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
gpt = models.Model(inputs=inputs, outputs=[outputs, attention_scores])
gpt.compile("adam", loss=[losses.SparseCategoricalCrossentropy(), None])

In [24]:
gpt.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 256)      │     2,580,480 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ [(None, None, 256),    │       658,688 │
│ (TransformerBlock)              │ (None, 2, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 10000)    │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,809,168 (22.16 MB)

 Trainable params: 5,809,168 (22.16 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
if LOAD_MODEL:
    custom_objects = {
        "TokenAndPositionEmbedding": TokenAndPositionEmbedding,
        "TransformerBlock": TransformerBlock,
    }
    if os.path.exists("./models/gpt.keras"):
        gpt = models.load_model(
            "./models/gpt.keras", custom_objects=custom_objects, compile=True
        )
    elif os.path.exists("./models/gpt"):
        gpt = models.load_model(
            "./models/gpt", custom_objects=custom_objects, compile=True
        )
    elif os.path.exists("./checkpoint/checkpoint.weights.h5"):
        gpt.load_weights("./checkpoint/checkpoint.weights.h5")


## 9. Train the Transformer <a name="train"></a>

In [26]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("wine review", max_tokens=80, temperature=1.0)

In [27]:
# Create directories for checkpoints, logs, and models
os.makedirs("./checkpoint", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./models", exist_ok=True)

# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [28]:
gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/5
4057/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 2.5975
generated text:
wine review : france : france other : sauvignon blanc : herbaceous and herbaceous , this is a ripe and delicious , with its balanced nature by crispness . fresh red - currant and acidity at this stage , is the case of light - easy and fruity . screwcap . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 338s 76ms/step - loss: 2.2468
Epoch 2/5
4056/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9759
generated text:
wine review : us : california : sangiovese : this has aromas 90 plus though a layer of fine tannic , bright acidity and supple tannins evident in this light - bodied cabernet - driven [UNK] with explosive cherry and blackberry flavors . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 1.9571
Epoch 3/5
4054/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.8997
generated text:
wine review : argentina : mendoza province : malbec : raspberry , cherry and plum aromas are solid and show an oaky aspect . this i

In [29]:
# Save the final model
os.makedirs("./models", exist_ok=True)
try:
    gpt.save("./models/gpt.keras")
except Exception:
    gpt.save("./models/gpt")


# 3. Generate text using the Transformer

In [30]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [31]:
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=1.0
)


generated text:
wine review : us : oregon : pinot noir : this appealing wine is more than most single varietal bottlings of this stretch , but it ' s completely shut down now . raspberry , spice , herb and cherry fruit lead into a long , spicy finish . 



In [32]:
info = text_generator.generate(
    "wine review : italy", max_tokens=80, temperature=0.5
)


generated text:
wine review : italy : piedmont : nebbiolo : this opens with aromas of black - skinned berry , underbrush , scorched earth and a whiff of brown spice . the ripe palate doles out fleshy wild cherry , raspberry jam , clove and star anise flavors . it ' s a bit shy but will be enjoyed soon . 



In [33]:
info = text_generator.generate(
    "wine review : germany", max_tokens=80, temperature=0.5
)
print_probs(info, vocab)


generated text:
wine review : germany : mosel : riesling : fresh apple and lemon notes abound in this delicate riesling . it ' s dry in style , but the palate is vivaciously fresh and spry , with a bristle of lemon - lime acidity . drink now through 2020 . 



::   	100.0%
-:   	0.0%
grosso:   	0.0%
giallo:   	0.0%
[UNK]:   	0.0%
--------



mosel:   	95.36000061035156%
rheinhessen:   	1.6299999952316284%
rheingau:   	1.5800000429153442%
pfalz:   	1.3899999856948853%
nahe:   	0.029999999329447746%
--------



::   	99.7300033569336%
-:   	0.27000001072883606%
blanc:   	0.0%
[UNK]:   	0.0%
grosso:   	0.0%
--------



riesling:   	100.0%
pinot:   	0.0%
grüner:   	0.0%
white:   	0.0%
scheurebe:   	0.0%
--------



::   	100.0%
-:   	0.0%
blanc:   	0.0%
grosso:   	0.0%
weissburgunder:   	0.0%
--------



... (38 outputs left out)

through:   	97.20999908447266%
.:   	2.6700000762939453%
and:   	0.03999999910593033%
or:   	0.029999999329447746%
,:   	0.029999999329447746%
--------



2020:   	72.05999755859375%
2019:   	16.979999542236328%
2021:   	5.179999828338623%
2018:   	2.6500000953674316%
2025:   	1.6299999952316284%
--------



.:   	99.87999725341797%
to:   	0.07999999821186066%
,:   	0.029999999329447746%
through:   	0.0%
and:   	0.0%
--------



:   	100.0%
drink:   	0.0%
enjoy:   	0.0%
it:   	0.0%
the:   	0.0%
--------

